# Notebook 5 — Medical Chatbot (Interactive)
Chat with your fine-tuned model in Colab.

In [ ]:
!pip install -q transformers peft

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import ipywidgets as widgets
from IPython.display import display, HTML

MODEL_PATH = "models/final_model"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH)
print("Model loaded. Ready to chat!")

In [ ]:
def get_response(question):
    prompt = f"""### Instruction:
{question}
### Response:
"""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=200)
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("### Response:")[-1].strip()

# ── Simple widget chat interface ──
out = widgets.Output()
txt = widgets.Text(placeholder="Ask a medical question...", layout=widgets.Layout(width="70%"))
btn = widgets.Button(description="Ask", button_style="primary")
history_out = widgets.Output()

def on_ask(b):
    q = txt.value.strip()
    if not q:
        return
    txt.value = ""
    with history_out:
        print(f"\nYou: {q}")
        ans = get_response(q)
        print(f"Bot: {ans}")

btn.on_click(on_ask)
display(HTML("<h3>🩺 Medical Q&A Chatbot</h3>"))
display(widgets.HBox([txt, btn]))
display(history_out)